In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/ramen-ratings.csv")
df.head()

,Review #,Brand,Variety,Style,Country,Stars,Top Ten
0,2580,New Touch,T's Restaurant Tantanmen,Cup,Japan,3.75,NaN
1,2579,Just Way,Noodles Spicy Hot Sesame Spicy Hot Sesame Guan...,Pack,Taiwan,1,NaN
2,2578,Nissin,Cup Noodles Chicken Vegetable,Cup,USA,2.25,NaN
3,2577,Wei Lih,GGE Ramen Snack Tomato Flavor,Pack,Taiwan,2.75,NaN
4,2576,Ching's Secret,Singapore Curry,Pack,India,3.75,NaN


In [3]:
df = df[df["Stars"] != "Unrated"].copy()
df["Stars"] = df["Stars"].astype(float)

df["name_len"] = df["Variety"].str.split().str.len()
df["brand_count"] = df["Brand"].map(df["Brand"].value_counts())

X = df.drop(columns=["Review #", "Stars", "Top Ten"])
y = df["Stars"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer([
    ('numeric1', num_pipe(impute='mean', poly=2, scaling='standard', transform='yeo-johnson'), ["name_len"]),
    ('numeric2', num_pipe(impute='median', poly=2, scaling='robust'), ["brand_count"]),
    ('categoric2', cat_pipe(encoder='onehot'), ["Brand", "Style", "Country"]),
    ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=3, stop_words='english'), "Variety"),
])

/home/ray/miniconda3/envs/lks-ai/lib/python3.9/site-packages/jcopml/pipeline/_pipeline.py:66: UserWarning: Transformer has default standardization, so the scaling argument is neglected
  warn("Transformer has default standardization, so the scaling argument is neglected")


In [5]:
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', RandomForestRegressor(n_jobs=-1, random_state=42))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.rf_params, cv=5, scoring='r2', n_iter=50, n_jobs=-1, verbose=1, random_state=42, error_score="raise")
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


KeyboardInterrupt: 

In [ ]:
save_model(model, "napoleon.pkl")